## Library imports

In [28]:
# libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import os
from pathlib import Path
from tkinter import Tk
from tkinter.filedialog import askdirectory

# Preprocessing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

# Metrics
from sklearn.metrics import confusion_matrix
from sklearn.metrics import roc_auc_score
from sklearn.metrics import classification_report
from sklearn.model_selection import cross_val_score
from sklearn.metrics import f1_score

# Models
import lightgbm as lgb
from catboost import CatBoostClassifier
from xgboost import XGBClassifier

# Hyperparameter tuning
from sklearn.model_selection import StratifiedKFold
import optuna

# exporting the model
import joblib

## Load the dataset

In [29]:
cleaned_data_path = Path("../streamlit/cleaned_data")
if not cleaned_data_path.exists():
    root = Tk()
    root.withdraw()
    cleaned_data_path = Path(askdirectory(title="Select the directory containing the cleaned data"))

In [30]:
train=pd.read_parquet(cleaned_data_path/'train_final.parquet')
test=pd.read_parquet(cleaned_data_path/'test_final.parquet')

In [31]:
for col in train.columns:
    print(col)

SK_ID_CURR
TARGET
NAME_CONTRACT_TYPE
CODE_GENDER
FLAG_OWN_CAR
FLAG_OWN_REALTY
CNT_CHILDREN
AMT_INCOME_TOTAL
AMT_CREDIT
AMT_ANNUITY
AMT_GOODS_PRICE
NAME_TYPE_SUITE
NAME_INCOME_TYPE
NAME_EDUCATION_TYPE
NAME_FAMILY_STATUS
NAME_HOUSING_TYPE
REGION_POPULATION_RELATIVE
DAYS_BIRTH
DAYS_EMPLOYED
DAYS_REGISTRATION
DAYS_ID_PUBLISH
FLAG_EMP_PHONE
FLAG_WORK_PHONE
FLAG_CONT_MOBILE
FLAG_PHONE
FLAG_EMAIL
OCCUPATION_TYPE
CNT_FAM_MEMBERS
REGION_RATING_CLIENT
REGION_RATING_CLIENT_W_CITY
WEEKDAY_APPR_PROCESS_START
HOUR_APPR_PROCESS_START
REG_REGION_NOT_LIVE_REGION
REG_REGION_NOT_WORK_REGION
LIVE_REGION_NOT_WORK_REGION
REG_CITY_NOT_LIVE_CITY
REG_CITY_NOT_WORK_CITY
LIVE_CITY_NOT_WORK_CITY
ORGANIZATION_TYPE
OBS_30_CNT_SOCIAL_CIRCLE
DEF_30_CNT_SOCIAL_CIRCLE
OBS_60_CNT_SOCIAL_CIRCLE
DEF_60_CNT_SOCIAL_CIRCLE
DAYS_LAST_PHONE_CHANGE
AMT_REQ_CREDIT_BUREAU_HOUR
AMT_REQ_CREDIT_BUREAU_DAY
AMT_REQ_CREDIT_BUREAU_WEEK
AMT_REQ_CREDIT_BUREAU_MON
AMT_REQ_CREDIT_BUREAU_QRT
AMT_REQ_CREDIT_BUREAU_YEAR
EXT_SOURCE_MEAN
HAS_BUI

### target separation

In [32]:
X = train.drop(columns=['SK_ID_CURR', 'TARGET'])
y = train['TARGET']

In [33]:
for col in X.select_dtypes(include='category').columns:
        X[col] = X[col].astype(str)

### endoding categorical features

In [34]:
cat_cols = X.select_dtypes(include='object').columns

len(cat_cols)

12

In [35]:
cat_cols

Index(['NAME_CONTRACT_TYPE', 'CODE_GENDER', 'FLAG_OWN_CAR', 'FLAG_OWN_REALTY',
       'NAME_TYPE_SUITE', 'NAME_INCOME_TYPE', 'NAME_EDUCATION_TYPE',
       'NAME_FAMILY_STATUS', 'NAME_HOUSING_TYPE', 'OCCUPATION_TYPE',
       'WEEKDAY_APPR_PROCESS_START', 'ORGANIZATION_TYPE'],
      dtype='object')

In [36]:
encoders = {}
for col in cat_cols:
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col])
    encoders[col] = le

### split the data into training and validation sets

In [37]:
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2,stratify= y ,random_state=42)

# model testing

### base model testing

In [38]:
model = lgb.LGBMClassifier(
    n_estimators      = 1000,
    learning_rate     = 0.05,
    num_leaves        = 63,
    max_depth         = -1,
    min_child_samples = 50,
    subsample         = 0.8,
    colsample_bytree  = 0.8,
    reg_alpha         = 0.1,
    reg_lambda        = 0.1,
    class_weight      = "balanced",   # handles class imbalance
    random_state      = 42,
    n_jobs            = -1,
    verbose           = -1,
)

model.fit(
    X_train, y_train,
    eval_set              = [(X_val, y_val)],
    callbacks             = [lgb.early_stopping(50, verbose=False),
                                lgb.log_evaluation(period=-1)],
)

pred_prob = model.predict_proba(X_val)[:, 1]
auc       = roc_auc_score(y_val, pred_prob)

In [39]:
auc

np.float64(0.7670645797540111)

In [40]:
pred = model.predict(X_val)

confusion_matrix(y_val, pred)

array([[47946,  8559],
       [ 2466,  2496]])

In [41]:
print(classification_report(y_val,pred))

              precision    recall  f1-score   support

           0       0.95      0.85      0.90     56505
           1       0.23      0.50      0.31      4962

    accuracy                           0.82     61467
   macro avg       0.59      0.68      0.60     61467
weighted avg       0.89      0.82      0.85     61467



In [42]:
feature_imp = pd.DataFrame({
    'Feature': X_train.columns,
    'Importance': model.feature_importances_
}).sort_values(
    by='Importance',
    ascending=False
)

feature_imp.tail(50)

,Feature,Importance
154,MAX_POS_DPD_DEF,116
4,CNT_CHILDREN,115
27,REGION_RATING_CLIENT_W_CITY,106
13,NAME_HOUSING_TYPE,101
141,AVG_RECEIVABLE,101
38,DEF_30_CNT_SOCIAL_CIRCLE,100
135,AVG_CC_DPD,99
45,AMT_REQ_CREDIT_BUREAU_MON,97
40,DEF_60_CNT_SOCIAL_CIRCLE,80
20,FLAG_WORK_PHONE,79


In [43]:
cv = StratifiedKFold(n_splits=5,shuffle=True,random_state=42)

In [44]:
cv_scores = cross_val_score(model,X,y,scoring='roc_auc',cv=cv,n_jobs=-1)

In [45]:
print(cv_scores)
print(cv_scores.mean())
print(cv_scores.std())

[0.77372652 0.77600069 0.76064352 0.76941889 0.77184689]
0.7703273018279407
0.005303810543844025


## hyperparameter tuning with optuna for lightgbm

In [46]:
def objective(trial):
    params={
        'objective': 'binary',
        'metric': 'auc',
        'boosting_type': 'gbdt',
        'class_weight': 'balanced',
        'n_estimators': trial.suggest_int('n_estimators', 300,2000),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        'num_leaves': trial.suggest_int('num_leaves', 20, 150),
        'max_depth': trial.suggest_int('max_depth', 3, 15),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-3, 10.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-3, 10.0, log=True),
        'random_state': 42,
        'n_jobs': -1,
        'verbose': -1,
    }
    model = lgb.LGBMClassifier(**params)
    model.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        callbacks=[lgb.early_stopping(50, verbose=False)],
    )
    pred_prob = model.predict_proba(X_val)[:, 1]
    auc = roc_auc_score(y_val, pred_prob)
    return auc

In [47]:
lgb_study = optuna.create_study(direction='maximize')
lgb_study.optimize(objective, n_trials=50)

[I 2026-06-02 12:52:54,447] A new study created in memory with name: no-name-888ecdf1-44ab-41b8-8398-b04ca93b0f92
[I 2026-06-02 12:53:16,485] Trial 0 finished with value: 0.7758949575931134 and parameters: {'n_estimators': 1845, 'learning_rate': 0.02525387364808817, 'num_leaves': 135, 'max_depth': 15, 'subsample': 0.8510968395039753, 'colsample_bytree': 0.8682555266107059, 'reg_alpha': 0.2681770635886934, 'reg_lambda': 0.07404719273299806}. Best is trial 0 with value: 0.7758949575931134.
[I 2026-06-02 12:53:55,459] Trial 1 finished with value: 0.7788267730602504 and parameters: {'n_estimators': 939, 'learning_rate': 0.01797643542324466, 'num_leaves': 90, 'max_depth': 12, 'subsample': 0.7755958823745761, 'colsample_bytree': 0.9723022590813227, 'reg_alpha': 7.469654869979819, 'reg_lambda': 1.1370118506683138}. Best is trial 1 with value: 0.7788267730602504.
[I 2026-06-02 12:54:08,160] Trial 2 finished with value: 0.7770847379113204 and parameters: {'n_estimators': 1619, 'learning_rate': 

In [48]:
lgb_study.best_params

{'n_estimators': 1800,
 'learning_rate': 0.011835754679655579,
 'num_leaves': 75,
 'max_depth': 13,
 'subsample': 0.9088890228641653,
 'colsample_bytree': 0.9080242776378811,
 'reg_alpha': 9.766642187212563,
 'reg_lambda': 2.845608939607423}

In [49]:
lgb_study.best_value

0.7792227994076992

## tunned lightgbm model

In [50]:
lgb_model = lgb.LGBMClassifier(
    **lgb_study.best_params,
    class_weight      = "balanced",
    random_state      = 42,
    n_jobs            = -1,
    verbose           = -1,
)

In [51]:
lgb_model.fit(
    X_train,
    y_train,
    eval_set=(X_val, y_val)
)

LGBMClassifier(class_weight='balanced', colsample_bytree=0.9080242776378811,
               learning_rate=0.011835754679655579, max_depth=13,
               n_estimators=1800, n_jobs=-1, num_leaves=75, random_state=42,
               reg_alpha=9.766642187212563, reg_lambda=2.845608939607423,
               subsample=0.9088890228641653, verbose=-1)

In [52]:
lgb_pred = lgb_model.predict_proba(X_val)[:,1]
lgb_auc = roc_auc_score(y_val, lgb_pred)
print(lgb_auc)

0.7791596239374294


### hyperparameter tuning with optuna for catboost

In [ ]:
# =========================================
# OPTUNA OBJECTIVE FUNCTION FOR CATBOOST
# =========================================

def objective(trial):

    params = {
        'loss_function': 'Logloss',
        'eval_metric': 'AUC',

        'iterations': trial.suggest_int('iterations',300,2000),

        'learning_rate': trial.suggest_float('learning_rate',0.01,0.1,log=True),

        'depth': trial.suggest_int('depth',4,10),

        'l2_leaf_reg': trial.suggest_float('l2_leaf_reg',1e-3,10.0,log=True),

        'subsample': trial.suggest_float('subsample',0.6,1.0),

        'random_strength': trial.suggest_float('random_strength',1e-3,10.0,log=True),

        'bagging_temperature': trial.suggest_float('bagging_temperature',0,10),

        'scale_pos_weight': trial.suggest_float('scale_pos_weight',1,10),

        'random_state': 42,

        'verbose': 0
    }

    # =====================================
    # MODEL
    # =====================================

    cat_model = CatBoostClassifier(**params)

    # =====================================
    # TRAIN
    # =====================================

    cat_model.fit(X_train,y_train,

        eval_set=(X_val, y_val),

        early_stopping_rounds=50,

        verbose=False
    )

    # =====================================
    # PREDICTION
    # =====================================

    pred_prob = cat_model.predict_proba(X_val)[:,1]

    # =====================================
    # ROC-AUC
    # =====================================

    auc = roc_auc_score(y_val,pred_prob)

    return auc

In [54]:
cat_study = optuna.create_study(direction='maximize')

cat_study.optimize(objective,n_trials=76)

[I 2026-06-02 13:18:41,352] A new study created in memory with name: no-name-ce8ffeb1-c8c2-44a3-8f09-62acb342163e
[I 2026-06-02 13:19:53,648] Trial 0 finished with value: 0.7738843526882531 and parameters: {'iterations': 1618, 'learning_rate': 0.01617997822308227, 'depth': 9, 'l2_leaf_reg': 0.08169360158950896, 'subsample': 0.6542666857802705, 'random_strength': 0.27496508098547967, 'bagging_temperature': 7.285412984395769, 'scale_pos_weight': 6.298208684047573}. Best is trial 0 with value: 0.7738843526882531.
[I 2026-06-02 13:20:15,327] Trial 1 finished with value: 0.7782347076610663 and parameters: {'iterations': 1919, 'learning_rate': 0.0940841727586411, 'depth': 4, 'l2_leaf_reg': 0.015119143828827704, 'subsample': 0.9353860550292014, 'random_strength': 0.3594296134464122, 'bagging_temperature': 4.514491348171898, 'scale_pos_weight': 6.9732881101265125}. Best is trial 1 with value: 0.7782347076610663.
[I 2026-06-02 13:21:16,306] Trial 2 finished with value: 0.7739466900037489 and pa

In [55]:
cat_study.best_params

{'iterations': 923,
 'learning_rate': 0.06246840050225421,
 'depth': 5,
 'l2_leaf_reg': 0.40763839648570827,
 'subsample': 0.6201912349614332,
 'random_strength': 1.0425028928591424,
 'bagging_temperature': 3.643906711290002,
 'scale_pos_weight': 3.5294845828216053}

In [56]:
cat_study.best_value

0.7812454701746904

In [57]:
# =========================================
# FINAL TUNED CATBOOST MODEL
# =========================================
cat_model = CatBoostClassifier(
    **cat_study.best_params,
    loss_function='Logloss',
    eval_metric='AUC',
    random_state=42,
    verbose=100
)

# =========================================
# TRAIN MODEL
# =========================================

cat_model.fit(

    X_train,
    y_train,

    eval_set=(X_val, y_val),

    early_stopping_rounds=50,

    verbose=False
)

# =========================================
# PREDICTIONS
# =========================================

cat_pred = cat_model.predict_proba(X_val)[:,1]

# =========================================
# ROC-AUC SCORE
# =========================================

cat_auc = roc_auc_score(y_val,cat_pred)

print("CatBoost ROC-AUC :", cat_auc)

CatBoost ROC-AUC : 0.7812454701746904


In [58]:
feature_imp = pd.DataFrame({
    'Feature': X_train.columns,
    'Importance': cat_model.feature_importances_
}).sort_values(
    by='Importance',
    ascending=False
)

feature_imp.tail(50)

,Feature,Importance
73,AVG_BUREAU_STATUS,0.176143
118,DPD_OVER_30_MEAN,0.175565
95,AVG_SEVERE_DPD_RATIO,0.175050
4,CNT_CHILDREN,0.172988
128,AVG_CC_BALANCE,0.161320
101,OVERDUE_PER_ACTIVE_LOAN,0.159812
141,AVG_RECEIVABLE,0.154330
13,NAME_HOUSING_TYPE,0.146403
135,AVG_CC_DPD,0.142030
9,NAME_TYPE_SUITE,0.140530


### checking if dropping 0 importance in both models improves performance

In [59]:
col_to_drop = feature_imp[feature_imp['Importance'] == 0]['Feature'].tolist()
X_train_reduced = X_train.drop(columns=col_to_drop)
X_val_reduced = X_val.drop(columns=col_to_drop)

In [60]:
cat_model.fit(

    X_train_reduced,
    y_train,

    eval_set=(X_val_reduced, y_val),

    early_stopping_rounds=50,

    verbose=False
)
cat_pred = cat_model.predict_proba(X_val_reduced)[:,1]
cat_auc = roc_auc_score(y_val,cat_pred)

print("CatBoost ROC-AUC :", cat_auc)

CatBoost ROC-AUC : 0.7809253806497739


### instead of helping it actually made it drop the roc_auc, so we will keep all features for the final model

### ensemble on two models
ensemble_pred = 0.5 * cat_pred + 0.5 * lgb_pred

In [61]:
blend_pred = (
    0.5 * lgb_pred
    +
    0.5 * cat_pred
)

In [62]:
blend_auc = roc_auc_score(y_val,blend_pred)

print("Blended ROC-AUC :", blend_auc)

Blended ROC-AUC : 0.7817484272382327


### FINAL xgboost model

In [63]:
# =========================================
# XGBOOST MODEL
# =========================================
xgb_model = XGBClassifier(
    objective='binary:logistic',
    eval_metric='auc',
    n_estimators=1000,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=0.1,
    scale_pos_weight=(
        y_train.value_counts()[0]
        /
        y_train.value_counts()[1]
    ),
    random_state=42,
    n_jobs=-1
)

# =========================================
# TRAIN XGBOOST
# =========================================

xgb_model.fit(

    X_train,
    y_train,

    eval_set=[(X_val, y_val)],

    verbose=False
)

# =========================================
# XGBOOST PREDICTIONS
# =========================================

xgb_pred = xgb_model.predict_proba(X_val)[:,1]

# =========================================
# XGBOOST ROC-AUC
# =========================================

xgb_auc = roc_auc_score(y_val,xgb_pred)

print("XGBoost ROC-AUC :", xgb_auc)

XGBoost ROC-AUC : 0.7698377343057212


In [64]:
# =========================================
# TRIPLE ENSEMBLE
# =========================================

ensemble_pred_prob = (
    0.5 * cat_pred
    +
    0.3 * lgb_pred
    +
    0.2 * xgb_pred
)

# =========================================
# ENSEMBLE ROC-AUC
# =========================================

ensemble_auc = roc_auc_score(y_val,ensemble_pred_prob)

print("Triple Ensemble ROC-AUC :", ensemble_auc)

Triple Ensemble ROC-AUC : 0.7819578161338802


In [65]:
ensemble_pred = (ensemble_pred_prob >= 0.5).astype(int)


In [66]:
print(classification_report(y_val,ensemble_pred))

              precision    recall  f1-score   support

           0       0.95      0.90      0.92     56505
           1       0.27      0.44      0.34      4962

    accuracy                           0.86     61467
   macro avg       0.61      0.67      0.63     61467
weighted avg       0.89      0.86      0.87     61467



In [67]:
print(confusion_matrix(y_val,ensemble_pred))

[[50641  5864]
 [ 2782  2180]]


### threshold tuning

In [68]:
thresholds = np.arange(0.10, 0.91, 0.01)

results = []

for threshold in thresholds:

    pred = (ensemble_pred_prob >= threshold).astype(int)
    f1 = f1_score(y_val,pred)
    results.append([threshold, f1])

results = pd.DataFrame(
    results,
    columns=['Threshold', 'F1']
)

results.sort_values(by='F1',ascending=False).head(10)

,Threshold,F1
41,0.51,0.335320
40,0.50,0.335230
43,0.53,0.335031
39,0.49,0.334699
42,0.52,0.334455
38,0.48,0.333669
37,0.47,0.332156
44,0.54,0.330959
35,0.45,0.330336
36,0.46,0.330299


In [69]:
best_threshold = 0.55

final_pred = (ensemble_pred_prob >= best_threshold).astype(int)

print(classification_report(y_val,final_pred))
print(confusion_matrix(y_val,final_pred))

              precision    recall  f1-score   support

           0       0.94      0.93      0.93     56505
           1       0.30      0.36      0.33      4962

    accuracy                           0.88     61467
   macro avg       0.62      0.64      0.63     61467
weighted avg       0.89      0.88      0.89     61467

[[52300  4205]
 [ 3161  1801]]


### final model comparison

In [70]:
comparison = pd.DataFrame({

    'Model':[
        'LightGBM',
        'CatBoost',
        'XGBoost',
        'Triple Ensemble'
    ],

    'ROC_AUC':[
        lgb_auc,
        cat_auc,
        xgb_auc,
        ensemble_auc
    ]
})

comparison.sort_values(
    by='ROC_AUC',
    ascending=False
)

,Model,ROC_AUC
3,Triple Ensemble,0.781958
1,CatBoost,0.780925
0,LightGBM,0.779160
2,XGBoost,0.769838


# Retrain models on full training data

In [71]:
lgb_final = lgb.LGBMClassifier(
    **lgb_study.best_params,
    class_weight      = "balanced",
    random_state      = 42,
    n_jobs            = -1,
    verbose           = -1,
)

cat_final = CatBoostClassifier(
    **cat_study.best_params,
    loss_function='Logloss',
    eval_metric='AUC',
    random_state=42,
    verbose=100
)

xgb_final = XGBClassifier(
    objective='binary:logistic',
    eval_metric='auc',
    n_estimators=1000,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=0.1,
    scale_pos_weight=(
        y.value_counts()[0]
        /
        y.value_counts()[1]
    ),
    random_state=42,
    n_jobs=-1
)

In [72]:
cat_final.fit(X, y,verbose=False)
lgb_final.fit(X, y)
xgb_final.fit(X, y)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.8, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='auc', feature_types=None,
              feature_weights=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=0.05, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=6,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=1000,
              n_jobs=-1, num_parallel_tree=None, ...)

In [73]:
ensemble_info = {

    'cat_weight': 0.5,

    'lgb_weight': 0.3,

    'xgb_weight': 0.2,

    'threshold': 0.5
}

In [74]:
model_metrics = {
    "roc_auc": ensemble_auc,
    "cat_weight": 0.5,
    "lgb_weight": 0.3,
    "xgb_weight": 0.2,
    "threshold": 0.5
}

## Exporting the models ,encoders and ensemble info

In [75]:
model_export_path = Path("../streamlit/models")
if not model_export_path.exists():
    root = Tk()
    root.withdraw()
    model_export_path = Path(askdirectory(title="Select the directory containing the cleaned data"))

In [76]:
joblib.dump(cat_final,model_export_path/'catboost_model.pkl')

joblib.dump(lgb_final,model_export_path/'lightgbm_model.pkl')

joblib.dump(xgb_final,model_export_path/'xgboost_model.pkl')

joblib.dump(encoders,model_export_path/'encoders.pkl')

joblib.dump(X.columns.tolist(),model_export_path/'feature_columns.pkl')

joblib.dump(ensemble_info,model_export_path/'ensemble_info.pkl')

joblib.dump(model_metrics,model_export_path / "model_metrics.pkl")

['..\\streamlit\\models\\model_metrics.pkl']